# Silver Quarantine Summary

This notebook reads all Silver quarantine tables and creates consolidated data quality summaries for rejected records and rejection reasons.

In [0]:
from pyspark.sql import functions as F

In [0]:
SILVER_BASE_PATH = (
    "abfss://silver@stnovacartdev.dfs.core.windows.net/"
    "olist"
)

display(dbutils.fs.ls(SILVER_BASE_PATH))
QUARANTINE_BASE_PATH = (
    "abfss://quarantine@stnovacartdev.dfs.core.windows.net/"
    "olist"
)
GOLD_QUALITY_BASE_PATH = (
    "abfss://gold@stnovacartdev.dfs.core.windows.net/"
    "olist/data_quality"
)

In [0]:
customers_quarantine_df = (
    spark.read
    .format("delta")
    .load(f"{QUARANTINE_BASE_PATH}/customers")
)

customers_quarantine_df.printSchema()
display(customers_quarantine_df.limit(10))

In [0]:
DATASETS = [
    "category_translation",
    "customers",
    "geolocation",
    "order_items",
    "order_payments",
    "order_reviews",
    "orders",
    "products",
    "sellers",
]

In [0]:
quarantine_summary_dfs = []

for dataset in DATASETS:
    quarantine_path = f"{QUARANTINE_BASE_PATH}/{dataset}"

    quarantine_df = (
        spark.read
        .format("delta")
        .load(quarantine_path)
    )

    summary_df = (
        quarantine_df
        .agg(
            F.count("*").alias("rejected_row_count"),
            F.countDistinct("_batch_id").alias("affected_batch_count"),
            F.min("_quarantined_at").alias("first_quarantined_at"),
            F.max("_quarantined_at").alias("last_quarantined_at"),
        )
        .withColumn("dataset_name", F.lit(dataset))
        .select(
            "dataset_name",
            "rejected_row_count",
            "affected_batch_count",
            "first_quarantined_at",
            "last_quarantined_at",
        )
    )

    quarantine_summary_dfs.append(summary_df)

In [0]:
silver_quarantine_summary_df = quarantine_summary_dfs[0]

for summary_df in quarantine_summary_dfs[1:]:
    silver_quarantine_summary_df = silver_quarantine_summary_df.unionByName(
        summary_df
    )

silver_quarantine_summary_df = (
    silver_quarantine_summary_df
    .withColumn(
        "quality_status",
        F.when(
            F.col("rejected_row_count") == 0,
            F.lit("PASS"),
        ).otherwise(F.lit("REVIEW")),
    )
    .withColumn(
        "_quality_reported_at",
        F.current_timestamp(),
    )
    .orderBy("dataset_name")
)

display(silver_quarantine_summary_df)

In [0]:
rejection_reason_dfs = []

for dataset in DATASETS:
    quarantine_df = (
        spark.read
        .format("delta")
        .load(f"{QUARANTINE_BASE_PATH}/{dataset}")
    )

    reason_summary_df = (
        quarantine_df
        .filter(F.col("_rejection_reason").isNotNull())
        .groupBy("_rejection_reason")
        .agg(
            F.count("*").alias("rejected_row_count"),
            F.countDistinct("_batch_id").alias("affected_batch_count"),
            F.min("_quarantined_at").alias("first_quarantined_at"),
            F.max("_quarantined_at").alias("last_quarantined_at"),
        )
        .withColumn("dataset_name", F.lit(dataset))
        .select(
            "dataset_name",
            F.col("_rejection_reason").alias("rejection_reason"),
            "rejected_row_count",
            "affected_batch_count",
            "first_quarantined_at",
            "last_quarantined_at",
        )
    )

    rejection_reason_dfs.append(reason_summary_df)

In [0]:
rejection_reason_summary_df = rejection_reason_dfs[0]

for reason_df in rejection_reason_dfs[1:]:
    rejection_reason_summary_df = (
        rejection_reason_summary_df.unionByName(reason_df)
    )

rejection_reason_summary_df = (
    rejection_reason_summary_df
    .withColumn("_quality_reported_at", F.current_timestamp())
    .orderBy(
        F.col("rejected_row_count").desc(),
        F.col("dataset_name"),
        F.col("rejection_reason"),
    )
)

display(rejection_reason_summary_df)

In [0]:
GOLD_QUALITY_BASE_PATH = (
    "abfss://gold@stnovacartdev.dfs.core.windows.net/"
    "olist/data_quality"
)

SILVER_QUARANTINE_SUMMARY_PATH = (
    f"{GOLD_QUALITY_BASE_PATH}/silver_quarantine_summary"
)

REJECTION_REASON_SUMMARY_PATH = (
    f"{GOLD_QUALITY_BASE_PATH}/rejection_reason_summary"
)

In [0]:
(
    silver_quarantine_summary_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_QUARANTINE_SUMMARY_PATH)
)

In [0]:
(
    rejection_reason_summary_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(REJECTION_REASON_SUMMARY_PATH)
)

In [0]:
saved_quarantine_summary_df = (
    spark.read
    .format("delta")
    .load(SILVER_QUARANTINE_SUMMARY_PATH)
)

saved_rejection_reason_summary_df = (
    spark.read
    .format("delta")
    .load(REJECTION_REASON_SUMMARY_PATH)
)

print(
    "Silver quarantine summary rows:",
    saved_quarantine_summary_df.count(),
)

print(
    "Rejection reason summary rows:",
    saved_rejection_reason_summary_df.count(),
)

display(saved_quarantine_summary_df)
display(saved_rejection_reason_summary_df)